# Lesson 10 — Final project: Duka Bora 2025 business review
**Brief from the owner:** *"Tell me how 2025 went: revenue and profit, which branches and products drive it, how customers pay, and what to fix."*

We'll use everything from lessons 1–9 in one clean pipeline.

In [1]:
import pandas as pd, numpy as np
orders    = pd.read_csv('data/orders.csv', parse_dates=['order_date'])
products  = pd.read_csv('data/products.csv')
customers = pd.read_csv('data/customers.csv')
mm        = pd.read_csv('data/mobile_money.csv', parse_dates=['timestamp'])

df = (orders
      .merge(products[['product_id', 'product_name', 'category', 'unit_cost']], on='product_id')
      .merge(customers[['customer_id', 'city', 'segment']], on='customer_id', how='left')
      .assign(revenue=lambda d: d.quantity * d.unit_price * (1 - d.discount_pct/100),
              cost=lambda d: d.quantity * d.unit_cost)
      .assign(profit=lambda d: d.revenue - d.cost,
              month=lambda d: d.order_date.dt.to_period('M')))
df.shape

(2400, 19)

## 1. Headline numbers

In [2]:
kpi = pd.Series({
    'Orders': len(df),
    'Revenue (TSh)': df.revenue.sum(),
    'Profit (TSh)': df.profit.sum(),
    'Margin': df.profit.sum() / df.revenue.sum(),
    'Avg order (TSh)': df.revenue.mean(),
    'Customers': df.customer_id.nunique(),
})
kpi.map(lambda v: f'{v:.1%}' if v < 1 else f'{v:,.0f}')

Orders                   2,400
Revenue (TSh)      319,634,300
Profit (TSh)        98,967,300
Margin                   31.0%
Avg order (TSh)        133,181
Customers                  300
dtype: str

## 2. Branch scorecard

In [3]:
branches = df.groupby('branch').agg(orders=('order_id', 'count'), revenue=('revenue', 'sum'), profit=('profit', 'sum'))
branches['margin'] = branches.profit / branches.revenue
branches['share'] = branches.revenue / branches.revenue.sum()
branches.sort_values('revenue', ascending=False).style.format({'revenue': '{:,.0f}', 'profit': '{:,.0f}', 'margin': '{:.1%}', 'share': '{:.1%}'})

,orders,revenue,profit,margin,share
branch,,,,,
Kariakoo,835,"107,063,150","33,706,150",31.5%,33.5%
Mlimani City,720,"98,931,100","29,938,100",30.3%,31.0%
Arusha CBD,458,"64,620,500","19,845,500",30.7%,20.2%
Mwanza Rock City,387,"49,019,550","15,477,550",31.6%,15.3%


## 3. Product mix: revenue vs profit

In [4]:
prod = df.groupby('product_name').agg(units=('quantity', 'sum'), revenue=('revenue', 'sum'), profit=('profit', 'sum'))
prod['margin'] = prod.profit / prod.revenue
prod.sort_values('profit', ascending=False).head(6).style.format({'revenue': '{:,.0f}', 'profit': '{:,.0f}', 'margin': '{:.1%}'})

,units,revenue,profit,margin
product_name,,,,
Smartphone 128GB,182,"112,807,500","18,167,500",16.1%
Bluetooth Earbuds,337,"27,523,000","11,347,000",41.2%
Wireless Mouse,959,"22,922,500","9,496,500",41.4%
"10,000mAh Power Bank",372,"19,629,500","8,469,500",43.1%
1TB External SSD,113,"26,088,000","7,443,000",28.5%
Over-ear Headphones,127,"19,496,000","7,431,000",38.1%


## 4. Seasonality

In [5]:
monthly = df.groupby('month')['revenue'].sum()
peak, low = monthly.idxmax(), monthly.idxmin()
print(f'Peak month {peak}: TSh {monthly.max()/1e6:.1f}m | Low month {low}: TSh {monthly.min()/1e6:.1f}m | ratio {monthly.max()/monthly.min():.1f}x')

Peak month 2025-12: TSh 45.4m | Low month 2025-02: TSh 16.8m | ratio 2.7x


## 5. Payments health check

In [6]:
pay = df.groupby('payment_method')['revenue'].agg(['count', 'sum'])
pay['share'] = pay['sum'] / pay['sum'].sum()
print(pay.style.format({'sum': '{:,.0f}', 'share': '{:.1%}'}).to_string())
failed = mm[mm.status != 'SUCCESS']
print(f'\nMobile-money issues: {len(failed)} txns ({len(failed)/len(mm):.1%}), TSh {failed.amount_tzs.sum():,.0f} to follow up')
print(failed.groupby(['provider', 'status']).size().unstack(fill_value=0))

 count sum share
payment_method   
Card 317 40,851,000 12.8%
Cash 701 98,999,150 31.0%
Mobile Money 1382 179,784,150 56.2%


Mobile-money issues: 226 txns (16.6%), TSh 28,015,900 to follow up
status        FAILED  REVERSED
provider                      
Airtel Money      21        25
HaloPesa           9         4
M-Pesa            69        48
Tigo Pesa         25        25


## 6. Export the report

In [7]:
with pd.ExcelWriter('duka_bora_2025_review.xlsx') as xl:
    kpi.to_frame('value').to_excel(xl, sheet_name='KPIs')
    branches.to_excel(xl, sheet_name='Branches')
    prod.sort_values('profit', ascending=False).to_excel(xl, sheet_name='Products')
    monthly.to_frame('revenue').to_excel(xl, sheet_name='Monthly')
    failed.to_excel(xl, sheet_name='Payment issues', index=False)
print('Saved duka_bora_2025_review.xlsx')

Saved duka_bora_2025_review.xlsx


## Your findings (write them like an analyst)
Use the numbers above to write 3–5 bullet points for the owner, e.g. *which branch leads, which product earns the most profit (not just revenue), how big December is, and how much mobile money needs following up.*

🎓 **Course complete!** You can now load, clean, reshape, combine, analyse, chart and report on real business data with pandas.